## Import necessary libraries

In [ ]:
import mdtraj as md
import time
import pandas as pd
import numpy as np

## Sec struture analysis

In [ ]:
# Function to display a countdown
def countdown_timer(seconds):
    for i in range(seconds, 0, -1):
        time.sleep(1)
        print(f'Countdown: {i} seconds remaining...', end='\r')

# Parameters
batch_size = 1  
num_batches = 10 // batch_size 

all_sec_struct = []

# Mapping of DSSP codes
dssp_to_structure = {
    'H': 'Helix',   # Alpha helix
    'B': 'Sheet',   # Beta bridge
    'E': 'Sheet',   # Extended strand
    'G': 'Helix',   # 310 helix
    'I': 'Helix',   # Pi helix
    'C': 'Coil',    # Coil
    'T': 'Coil',    # Turn
    'S': 'Coil',    # Bend
    'NA': 'Not Assigned'  # Not assigned
}

for batch in range(num_batches):
    start_index = batch * batch_size
    end_index = start_index + batch_size
    xtc_files = [f'{i}.xtc' for i in range(start_index, min(end_index, 10))] 

    # Countdown before starting the batch
    print(f"\nProcessing batch {batch + 1}/{num_batches} in 5 seconds...")
    countdown_timer(5)

    print("Loading trajectory files...")

    start_time = time.time() 

    traj = md.load(xtc_files, top='topology.pdb')

    load_time = time.time() - start_time
    print(f"Batch {batch + 1} loaded! Time taken: {load_time:.2f} seconds.")

    print("Computing DSSP secondary structure in 5 seconds...")
    countdown_timer(5)

    start_time = time.time()

    sec_struct = md.compute_dssp(traj)

    dssp_time = time.time() - start_time
    print(f"DSSP secondary structure computed for batch {batch + 1}! Time taken: {dssp_time:.2f} seconds.")

    # Store the computed secondary structure
    all_sec_struct.append(sec_struct)

# Convert the list of arrays into a single 2D array
all_sec_struct = np.concatenate(all_sec_struct, axis=0)

sec_str_df = pd.DataFrame(all_sec_struct, columns=[f'Residue_{i+1}' for i in range(all_sec_struct.shape[1])])

# Map DSSP codes to meaningful secondary structure types
sec_str_df = sec_str_df.applymap(lambda x: dssp_to_structure.get(x, 'Not Assigned'))

structure_types = ['Helix', 'Sheet', 'Coil', 'Not Assigned']
structure_count_df = pd.DataFrame(index=sec_str_df.index, columns=structure_types)

for structure in structure_types:
    structure_count_df[structure] = sec_str_df.apply(lambda row: (row == structure).sum(), axis=1)

structure_percentage_df = structure_count_df.div(16).multiply(100)  
# Display per-frame structure percentages
print("Percentage of each structure type per frame:")
print(structure_percentage_df)

structure_ratio_df = pd.DataFrame(index=sec_str_df.index, columns=['Folded/Unfolded'])

# Folded includes Helix + Sheet; Unfolded includes Coil
structure_ratio_df['Folded/Unfolded'] = (structure_count_df['Helix'] + structure_count_df['Sheet']) / structure_count_df['Coil']

# Display the folded-to-unfolded ratio per frame
print("Folded to Unfolded Ratio per frame:")
print(structure_ratio_df)

In [ ]:
structure_percentage_df.to_csv('structure_percentage.csv', index =False)

### Importing files

In [ ]:
kmean_trj = pd.read_csv('auctoencoder_clustering.csv')

In [ ]:
merged_df = pd.merge(kmean_trj, structure_percentage_df, left_index=True, right_index=True)

# Step 2: Calculate Averages for Each Cluster
cluster_averages = merged_df.groupby('cluster')[['Helix', 'Sheet', 'Coil']].mean()

# Step 3: Display Results
print("Merged DataFrame:")
print(merged_df)

print("\nCluster Averages:")
print(cluster_averages)